# Hand Font — Google Colab 최소 시연본

**포함된 기능:** 회원가입 · 로그인 · 필기 업로드 · OpenCV 기본 분석 · 내 노트 저장/수정/삭제 · PNG 다운로드

- 백엔드: **FastAPI** (공식 Jinja2 템플릿 방식), SQLite, Starlette SessionMiddleware
- 화면: **Bootstrap 5.3 공식 CDN 스타일**, 필요한 화면 4개만 구성
- 분석: **OpenCV 실제 이미지 보정**. **YOLO / PaddleOCR / UniMERNet 모델은 아직 미연결**이며, AI 분석 결과로 소개하면 안 됩니다.
- 저장: `USE_GOOGLE_DRIVE=True`이면 계정 DB와 노트 이미지가 Google Drive `MyDrive/HandFontDemo`에 저장됩니다.
- Colab 공식 포트 공개 기능을 사용하며, 서버 창은 **노트북 실행자만 접근 가능**합니다. 공개 배포용 호스팅이 아닙니다.

위에서 아래로 **모든 셀 실행**을 누르면 됩니다. 마지막 셀이 사이트 창을 엽니다.

> 주의: Colab 무료 관리형 런타임은 웹 UI 중심으로 사용하다 종료될 수 있으며, 장시간 웹 서비스를 운영하는 용도로 적합하지 않습니다. 실서비스 비밀번호는 재사용하지 마세요.


In [ ]:
# 1. 패키지 설치 (Colab에서 실행)
%pip -q install 'fastapi>=0.110' 'uvicorn>=0.29' 'jinja2>=3.1' 'python-multipart>=0.0.9' 'itsdangerous>=2.1' 'pillow>=10' 'opencv-python-headless>=4.8' 'numpy>=1.24' 'requests>=2.31'
print('패키지 준비 완료')

패키지 준비 완료


In [ ]:
# 2. Google Drive 저장 옵션 & 프로젝트 폴더 준비
#@title 데이터 저장 설정
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
import os
from pathlib import Path
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HANDFONT_DATA_DIR'] = '/content/drive/MyDrive/HandFontDemo'
else:
    os.environ['HANDFONT_DATA_DIR'] = '/content/HandFontDemo'  # 런타임 초기화 시 삭제됨

Path('/content/handfont_demo/templates').mkdir(parents=True, exist_ok=True)
os.chdir('/content/handfont_demo')
print('노트/DB 저장 위치:', os.environ['HANDFONT_DATA_DIR'])

Mounted at /content/drive
노트/DB 저장 위치: /content/drive/MyDrive/HandFontDemo


## 3. FastAPI 및 공식 템플릿 기반 화면 코드
아래 셀들은 서버 파일과 HTML 템플릿을 자동으로 작성합니다. 직접 수정해도 됩니다.

In [ ]:
%%writefile app.py
"""Hand Font minimal classroom demo (FastAPI + SQLite + OpenCV)."""
from __future__ import annotations

import hashlib
import hmac
import io
import json
import os
import re
import secrets
import sqlite3
import uuid
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.responses import FileResponse, RedirectResponse
from fastapi.templating import Jinja2Templates
from PIL import Image, ImageOps, UnidentifiedImageError
from starlette.middleware.sessions import SessionMiddleware

BASE = Path(__file__).resolve().parent
DATA = Path(os.environ.get("HANDFONT_DATA_DIR", str(BASE / "data")))
IMAGES = DATA / "images"
DB = DATA / "handfont.db"
DATA.mkdir(parents=True, exist_ok=True)
IMAGES.mkdir(parents=True, exist_ok=True)

KEY_FILE = DATA / ".session_key"
if not KEY_FILE.exists():
    KEY_FILE.write_text(secrets.token_hex(32), encoding="utf-8")

app = FastAPI(title="Hand Font DEMO", version="0.1.0")
app.add_middleware(SessionMiddleware, secret_key=KEY_FILE.read_text().strip(), same_site="lax")
templates = Jinja2Templates(directory=str(BASE / "templates"))


def db():
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    return con


def init_db():
    with db() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
          id INTEGER PRIMARY KEY AUTOINCREMENT,
          username TEXT NOT NULL UNIQUE,
          password_hash TEXT NOT NULL,
          created_at TEXT NOT NULL
        )""")
        conn.execute("""CREATE TABLE IF NOT EXISTS notes (
          id INTEGER PRIMARY KEY AUTOINCREMENT,
          user_id INTEGER NOT NULL,
          title TEXT NOT NULL,
          subject TEXT NOT NULL,
          original TEXT NOT NULL,
          processed TEXT,
          status TEXT NOT NULL DEFAULT 'uploaded',
          stats TEXT,
          created_at TEXT NOT NULL,
          FOREIGN KEY(user_id) REFERENCES users(id)
        )""")


init_db()


def password_hash(password: str) -> str:
    salt = secrets.token_bytes(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 260_000)
    return f"{salt.hex()}:{digest.hex()}"


def check_password(password: str, stored: str) -> bool:
    try:
        salt, digest = stored.split(":", 1)
        test = hashlib.pbkdf2_hmac("sha256", password.encode(), bytes.fromhex(salt), 260_000)
        return hmac.compare_digest(test, bytes.fromhex(digest))
    except (ValueError, TypeError):
        return False


def current_user(request: Request):
    user_id = request.session.get("user_id")
    if not isinstance(user_id, int):
        return None
    with db() as conn:
        return conn.execute("SELECT id, username FROM users WHERE id=?", (user_id,)).fetchone()


def get_note(note_id: int, user_id: int):
    with db() as conn:
        note = conn.execute("SELECT * FROM notes WHERE id=? AND user_id=?", (note_id, user_id)).fetchone()
    if note is None:
        raise HTTPException(status_code=404, detail="노트를 찾을 수 없어.")
    return note


def view(request: Request, name: str, **data):
    return templates.TemplateResponse(request=request, name=name, context={"user": current_user(request), **data})


def require_login(request: Request):
    return current_user(request)


def go(location: str = "/notes"):
    return RedirectResponse(location, status_code=303)


@app.get("/health")
def health():
    return {"ok": True, "mode": "opencv_demo", "ai_model_connected": False}


@app.get("/")
def index(request: Request):
    return go("/notes" if current_user(request) else "/login")


@app.get("/signup")
def signup_form(request: Request):
    return view(request, "auth.html", mode="signup", error="")


@app.post("/signup")
def signup(request: Request, username: str = Form(...), password: str = Form(...)):
    username = username.strip()
    if not re.fullmatch(r"[a-zA-Z0-9_]{3,24}", username):
        return view(request, "auth.html", mode="signup", error="아이디는 영문·숫자·밑줄 3~24자만 가능해.")
    if len(password) < 6:
        return view(request, "auth.html", mode="signup", error="비밀번호를 6자 이상 입력해.")
    try:
        with db() as conn:
            cur = conn.execute("INSERT INTO users(username, password_hash, created_at) VALUES (?,?,?)",
                               (username, password_hash(password), datetime.now().isoformat(timespec="seconds")))
            user_id = cur.lastrowid
    except sqlite3.IntegrityError:
        return view(request, "auth.html", mode="signup", error="이미 사용 중인 아이디야.")
    request.session["user_id"] = user_id
    return go()


@app.get("/login")
def login_form(request: Request):
    return view(request, "auth.html", mode="login", error="")


@app.post("/login")
def login(request: Request, username: str = Form(...), password: str = Form(...)):
    with db() as conn:
        user = conn.execute("SELECT * FROM users WHERE username=?", (username.strip(),)).fetchone()
    if user is None or not check_password(password, user["password_hash"]):
        return view(request, "auth.html", mode="login", error="아이디 또는 비밀번호를 확인해.")
    request.session["user_id"] = user["id"]
    return go()


@app.post("/logout")
def logout(request: Request):
    request.session.clear()
    return go("/login")


@app.get("/notes")
def notes(request: Request):
    user = require_login(request)
    if not user:
        return go("/login")
    with db() as conn:
        rows = conn.execute("SELECT * FROM notes WHERE user_id=? ORDER BY id DESC", (user["id"],)).fetchall()
    return view(request, "notes.html", notes=rows)


@app.get("/upload")
def upload_form(request: Request):
    if not require_login(request):
        return go("/login")
    return view(request, "upload.html", error="")


@app.post("/upload")
async def upload(request: Request, title: str = Form(...), subject: str = Form("기타"), file: UploadFile = File(...)):
    user = require_login(request)
    if not user:
        return go("/login")
    title = title.strip()[:60]
    subject = subject.strip()[:30] or "기타"
    raw = await file.read(8 * 1024 * 1024 + 1)
    if not title:
        return view(request, "upload.html", error="노트 제목을 입력해.")
    if len(raw) > 8 * 1024 * 1024:
        return view(request, "upload.html", error="이미지는 8MB 이하만 올릴 수 있어.")
    try:
        # Validated image data is decoded and re-saved: filenames supplied by users are never used as paths.
        with Image.open(io.BytesIO(raw)) as im:
            if im.format not in ("JPEG", "PNG", "WEBP"):
                raise ValueError("format")
            if im.width * im.height > 16_000_000:
                return view(request, "upload.html", error="이미지 해상도가 너무 커. 최대 1,600만 픽셀이야.")
            converted = ImageOps.exif_transpose(im).convert("RGB")
            filename = uuid.uuid4().hex + ".png"
            target = IMAGES / filename
            converted.save(target, format="PNG", optimize=True)
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError):
        return view(request, "upload.html", error="JPG, PNG, WebP 이미지만 업로드해.")
    with db() as conn:
        cur = conn.execute("""INSERT INTO notes(user_id,title,subject,original,created_at)
            VALUES (?,?,?,?,?)""", (user["id"], title, subject, filename, datetime.now().strftime("%Y-%m-%d %H:%M")))
        note_id = cur.lastrowid
    return go(f"/notes/{note_id}")


def run_opencv_demo(source: Path, target: Path, mode: str):
    """Real image processing demo, not a handwriting reconstruction AI model."""
    image = cv2.imread(str(source))
    if image is None:
        raise ValueError("이미지를 읽을 수 없어.")
    height, width = image.shape[:2]
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    background = cv2.GaussianBlur(gray, (0, 0), sigmaX=22)
    leveled = cv2.divide(gray, background, scale=255)
    cleaned = cv2.fastNlMeansDenoising(leveled, None, h=5)
    if mode == "bw":
        result = cv2.adaptiveThreshold(cleaned, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 41, 13)
    else:
        result = cv2.createCLAHE(clipLimit=1.3, tileGridSize=(8, 8)).apply(cleaned)
    if not cv2.imwrite(str(target), result):
        raise ValueError("보정 이미지를 저장할 수 없어.")
    return {"width": width, "height": height, "mode": "흑백 대비" if mode == "bw" else "기본 보정",
            "steps": "조명 균일화 · 노이즈 완화 · 대비 보정", "ai": "AI 모델 미연결 (OpenCV 데모)"}


@app.get("/notes/{note_id}")
def note_detail(request: Request, note_id: int):
    user = require_login(request)
    if not user:
        return go("/login")
    note = get_note(note_id, user["id"])
    stats = json.loads(note["stats"]) if note["stats"] else None
    return view(request, "detail.html", note=note, stats=stats)


@app.post("/notes/{note_id}/analyze")
def analyze(request: Request, note_id: int, mode: str = Form("clean")):
    user = require_login(request)
    if not user:
        return go("/login")
    note = get_note(note_id, user["id"])
    if mode not in ("clean", "bw"):
        raise HTTPException(status_code=400, detail="지원하지 않는 분석 모드야.")
    filename = f"{uuid.uuid4().hex}.png"
    stats = run_opencv_demo(IMAGES / note["original"], IMAGES / filename, mode)
    with db() as conn:
        conn.execute("UPDATE notes SET processed=?, status='processed', stats=? WHERE id=? AND user_id=?",
                     (filename, json.dumps(stats, ensure_ascii=False), note_id, user["id"]))
    # Remove old generated version if replacing it.
    if note["processed"] and note["processed"] != filename:
        (IMAGES / note["processed"]).unlink(missing_ok=True)
    return go(f"/notes/{note_id}")


@app.post("/notes/{note_id}/rename")
def rename(request: Request, note_id: int, title: str = Form(...), subject: str = Form("기타")):
    user = require_login(request)
    if not user:
        return go("/login")
    get_note(note_id, user["id"])
    if not title.strip():
        raise HTTPException(400, "제목이 비어 있어.")
    with db() as conn:
        conn.execute("UPDATE notes SET title=?, subject=? WHERE id=? AND user_id=?",
                     (title.strip()[:60], subject.strip()[:30] or "기타", note_id, user["id"]))
    return go(f"/notes/{note_id}")


@app.post("/notes/{note_id}/delete")
def delete_note(request: Request, note_id: int):
    user = require_login(request)
    if not user:
        return go("/login")
    note = get_note(note_id, user["id"])
    with db() as conn:
        conn.execute("DELETE FROM notes WHERE id=? AND user_id=?", (note_id, user["id"]))
    for filename in (note["original"], note["processed"]):
        if filename:
            (IMAGES / filename).unlink(missing_ok=True)
    return go()


@app.get("/media/{note_id}/{kind}")
def media(request: Request, note_id: int, kind: str):
    user = require_login(request)
    if not user:
        raise HTTPException(status_code=401, detail="로그인이 필요해.")
    note = get_note(note_id, user["id"])
    if kind not in ("original", "processed") or not note[kind]:
        raise HTTPException(status_code=404)
    filepath = IMAGES / note[kind]
    if not filepath.is_file():
        raise HTTPException(status_code=404)
    return FileResponse(filepath, media_type="image/png", headers={"Cache-Control": "no-store"})


@app.get("/notes/{note_id}/download")
def download(request: Request, note_id: int):
    user = require_login(request)
    if not user:
        raise HTTPException(status_code=401)
    note = get_note(note_id, user["id"])
    filename = note["processed"] or note["original"]
    path = IMAGES / filename
    if not path.is_file():
        raise HTTPException(status_code=404)
    return FileResponse(path, media_type="image/png", filename=f"handfont_note_{note_id}.png")


Writing app.py


In [ ]:
%%writefile templates/base.html
<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Hand Font · Demo</title>
  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.9/dist/css/bootstrap.min.css" rel="stylesheet" integrity="sha384-yEPZ7jP9S0QKiUQ0m3jjsO4ProIW7wvR9gXkI7JOGa2XVsw53Yke2bsdkBbjmmiX" crossorigin="anonymous">
  <style>
    body {background:#f7f8fa;color:#202329;font-family:system-ui,-apple-system,"Noto Sans KR",sans-serif;}
    .navbar {background:#fff;border-bottom:1px solid #e7e9ec;}
    .brand {letter-spacing:-.04em;font-weight:750;color:#1b2638;text-decoration:none;}
    .app-container {max-width:990px;margin:48px auto 80px;}
    .card {border:1px solid #e5e8ec;border-radius:14px;box-shadow:0 3px 14px rgba(20,28,40,.025);}
    .btn {border-radius:8px;}
    .btn-dark {background:#24354d;border-color:#24354d;}
    .muted {color:#737b86;}
    .preview {width:100%;max-height:510px;object-fit:contain;background:#f3f5f7;border:1px solid #e5e8ec;border-radius:8px;}
    .note-card:hover {border-color:#bfcbd9;}
    .small-label {font-size:.82rem;color:#747d88;}
    @media(max-width:767px){.app-container{margin:26px auto 50px;}}
  </style>
</head>
<body>
  <nav class="navbar navbar-expand-sm py-3">
    <div class="container" style="max-width:990px">
      <a class="brand fs-5" href="/">Hand Font <span class="small fw-normal text-secondary">/ demo</span></a>
      {% if user %}
        <div class="d-flex align-items-center gap-3">
          <a class="text-decoration-none text-secondary small" href="/notes">내 노트</a>
          <a class="text-decoration-none text-secondary small" href="/upload">업로드</a>
          <form action="/logout" method="post" class="m-0"><button class="btn btn-sm btn-outline-secondary">로그아웃</button></form>
        </div>
      {% endif %}
    </div>
  </nav>
  <main class="container app-container">{% block content %}{% endblock %}</main>
</body>
</html>


Writing templates/base.html


In [ ]:
%%writefile templates/auth.html
{% extends "base.html" %}
{% block content %}
<div class="row justify-content-center"><div class="col-md-6 col-lg-5">
  <div class="mb-4"><h1 class="h3 fw-bold">{% if mode == 'signup' %}회원가입{% else %}로그인{% endif %}</h1>
  <p class="muted mb-0">필기를 업로드하고 나만의 노트로 정리해.</p></div>
  <div class="card p-4">
    {% if error %}<div class="alert alert-danger py-2 small">{{ error }}</div>{% endif %}
    <form method="post" action="/{{ mode }}">
      <label for="username" class="form-label">아이디</label>
      <input class="form-control mb-3" id="username" name="username" placeholder="영문, 숫자, _" minlength="3" maxlength="24" required autocomplete="username">
      <label for="password" class="form-label">비밀번호</label>
      <input class="form-control mb-4" type="password" id="password" name="password" minlength="6" placeholder="6자 이상" required autocomplete="{% if mode == 'signup' %}new-password{% else %}current-password{% endif %}">
      <button class="btn btn-dark w-100 py-2" type="submit">{% if mode == 'signup' %}계정 만들기{% else %}로그인{% endif %}</button>
    </form>
    <div class="small text-center text-secondary mt-3">
      {% if mode == 'signup' %}이미 계정이 있어? <a href="/login">로그인</a>{% else %}처음 이용해? <a href="/signup">회원가입</a>{% endif %}
    </div>
  </div>
  <p class="small text-secondary mt-3">학습용 시연 서비스 · 실제 서비스 계정 비밀번호를 재사용하지 마.</p>
</div></div>
{% endblock %}


Writing templates/auth.html


In [ ]:
%%writefile templates/notes.html
{% extends "base.html" %}
{% block content %}
<div class="d-flex justify-content-between align-items-center gap-3 mb-4">
  <div><h1 class="h3 fw-bold mb-1">내 노트</h1><p class="muted mb-0">{{ user.username }}의 필기 보관함</p></div>
  <a class="btn btn-dark" href="/upload">+ 필기 업로드</a>
</div>
{% if not notes %}
  <div class="card p-5 text-center"><h2 class="h5">저장된 노트가 없어</h2><p class="muted">첫 필기 사진을 업로드해 봐.</p>
  <div><a href="/upload" class="btn btn-outline-dark">이미지 업로드</a></div></div>
{% else %}
  <div class="row g-3">
  {% for note in notes %}
    <div class="col-md-6"><a class="text-decoration-none text-dark" href="/notes/{{ note.id }}"><div class="card note-card p-3 h-100">
      <div class="d-flex gap-3 align-items-center">
        <img src="/media/{{ note.id }}/{{ 'processed' if note.processed else 'original' }}" alt="필기 미리보기" style="width:100px;height:100px;object-fit:cover;background:#f3f3f3;border-radius:7px;">
        <div class="overflow-hidden"><div class="small-label">{{ note.subject }}</div><div class="fw-semibold text-truncate">{{ note.title }}</div>
          <div class="small text-secondary mt-1">{{ note.created_at }}</div>
          <div class="small mt-2">{{ '기본 분석 완료' if note.processed else '업로드됨' }}</div>
        </div>
      </div>
    </div></a></div>
  {% endfor %}
  </div>
{% endif %}
{% endblock %}


Writing templates/notes.html


In [ ]:
%%writefile templates/upload.html
{% extends "base.html" %}
{% block content %}
<a href="/notes" class="small text-decoration-none text-secondary">← 내 노트</a>
<h1 class="h3 fw-bold mt-3 mb-1">필기 업로드</h1>
<p class="muted mb-4">이미지를 추가한 다음 보정 결과를 확인할 수 있어.</p>
<div class="card p-4" style="max-width:690px">
  {% if error %}<div class="alert alert-danger py-2">{{ error }}</div>{% endif %}
  <form action="/upload" method="post" enctype="multipart/form-data">
    <label for="title" class="form-label">노트 제목</label>
    <input class="form-control mb-3" id="title" name="title" placeholder="예: 수학 - 삼각함수 필기" maxlength="60" required>
    <label for="subject" class="form-label">과목</label>
    <select class="form-select mb-3" id="subject" name="subject">
      <option>국어</option><option>수학</option><option>영어</option><option>과학</option><option>사회</option><option selected>기타</option>
    </select>
    <label for="file" class="form-label">필기 이미지</label>
    <input class="form-control mb-2" id="file" type="file" name="file" accept="image/png,image/jpeg,image/webp" required>
    <p class="small text-secondary mb-4">JPG / PNG / WebP · 8MB 이하</p>
    <button type="submit" class="btn btn-dark px-4">저장하고 계속</button>
  </form>
</div>
{% endblock %}


Writing templates/upload.html


In [ ]:
%%writefile templates/detail.html
{% extends "base.html" %}
{% block content %}
<a href="/notes" class="small text-decoration-none text-secondary">← 내 노트</a>
<div class="d-flex flex-wrap justify-content-between gap-2 align-items-center mt-3 mb-4">
  <div><div class="small-label">{{ note.subject }} · {{ note.created_at }}</div><h1 class="h3 fw-bold mb-0">{{ note.title }}</h1></div>
  <a href="/notes/{{ note.id }}/download" class="btn btn-outline-dark">PNG 다운로드</a>
</div>
<div class="row g-3 mb-3">
  <div class="col-md-6"><div class="card p-3 h-100"><h2 class="h6 fw-bold mb-3">원본 필기</h2>
    <img class="preview" src="/media/{{ note.id }}/original" alt="원본 필기"></div></div>
  <div class="col-md-6"><div class="card p-3 h-100"><h2 class="h6 fw-bold mb-3">보정 결과</h2>
  {% if note.processed %}
    <img class="preview" src="/media/{{ note.id }}/processed" alt="보정 결과">
  {% else %}<div class="border rounded bg-light d-flex justify-content-center align-items-center text-secondary" style="min-height:300px">아직 분석하지 않았어.</div>{% endif %}</div></div>
</div>
<div class="card p-4 mb-3">
  <h2 class="h6 fw-bold">기본 이미지 분석</h2>
  <p class="small muted">현재는 OpenCV 기반 조명·노이즈·대비 보정 데모야. YOLO / PaddleOCR / UniMERNet은 아직 연결되지 않았어.</p>
  <form action="/notes/{{ note.id }}/analyze" method="post" class="d-flex flex-wrap align-items-end gap-2">
    <div><label class="form-label small" for="mode">보정 방식</label>
    <select class="form-select" name="mode" id="mode"><option value="clean">기본 보정 (회색조)</option><option value="bw">흑백 대비 강조</option></select></div>
    <button class="btn btn-dark" type="submit">분석 및 보정 실행</button>
  </form>
  {% if stats %}
    <div class="alert alert-light border mt-3 mb-0 small"><strong>처리 완료</strong><div class="mt-1">이미지: {{ stats.width }} × {{ stats.height }}px · {{ stats.mode }}</div>
    <div>{{ stats.steps }}</div><div class="text-secondary">{{ stats.ai }}</div></div>
  {% endif %}
</div>
<div class="card p-4">
  <h2 class="h6 fw-bold">노트 관리</h2>
  <form action="/notes/{{ note.id }}/rename" method="post" class="row g-2 align-items-end mb-3">
    <div class="col-md-7"><label class="form-label small">제목 변경</label><input class="form-control" name="title" value="{{ note.title }}" maxlength="60" required></div>
    <div class="col-md-3"><label class="form-label small">과목</label><input class="form-control" name="subject" value="{{ note.subject }}" maxlength="30" required></div>
    <div class="col-md-2"><button class="btn btn-outline-dark w-100">저장</button></div>
  </form>
  <form action="/notes/{{ note.id }}/delete" method="post" onsubmit="return confirm('노트를 삭제할까?');">
    <button class="btn btn-sm btn-outline-danger">노트 삭제</button>
  </form>
</div>
{% endblock %}


Writing templates/detail.html


In [ ]:
# 4. FastAPI 시작 + Colab 공식 서버 창 열기
import subprocess, sys, time, requests
from google.colab import output

# 재실행 시 기존 프로세스만 정리
if 'handfont_server' in globals() and handfont_server.poll() is None:
    handfont_server.terminate()
    try:
        handfont_server.wait(timeout=4)
    except subprocess.TimeoutExpired:
        handfont_server.kill()

server_log = open('/content/handfont_demo/server.log', 'w', encoding='utf-8')
handfont_server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/handfont_demo',
    stdout=server_log, stderr=subprocess.STDOUT,
)

for _ in range(35):
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=1).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print(open('/content/handfont_demo/server.log', encoding='utf-8').read()[-3000:])
    raise RuntimeError('서버 실행에 실패했어. 위 로그를 확인해.')

print('서버 실행 완료! 아래 기능으로 사이트 창을 열게.')
print('API 상태:', requests.get('http://127.0.0.1:8000/health').json())
output.serve_kernel_port_as_window(8000)

서버 실행 완료! 아래 기능으로 사이트 창을 열게.
API 상태: {'ok': True, 'mode': 'opencv_demo', 'ai_model_connected': False}
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## 테스트 순서
1. 새 창에서 **회원가입** → 자동 로그인
2. **필기 업로드**에서 제목, 과목, PNG/JPG 이미지를 등록
3. 해당 노트에서 **분석 및 보정 실행** 클릭 → 원본/보정본 비교
4. **PNG 다운로드**, 노트 이름 변경, 삭제 등 확인
5. **로그아웃** → 다시 **로그인** → 노트가 유지되는지 확인

**연결 포인트:** `app.py`의 `run_opencv_demo(...)`를 YOLO·PaddleOCR·UniMERNet 추론 함수로 교체하거나 호출하도록 바꾸면 됩니다. 지금은 문자를 추출하거나 필체 자체를 AI로 보정하지 않습니다.

**개발 참고:** [FastAPI 공식 Jinja2 템플릿](https://fastapi.tiangolo.com/advanced/templates/) · [FastAPI 파일 업로드](https://fastapi.tiangolo.com/tutorial/request-forms-and-files/) · [Bootstrap 5.3 시작하기](https://getbootstrap.com/docs/5.3/getting-started/introduction/) · [Colab 서버 창 예제](https://colab.research.google.com/notebooks/snippets/advanced_outputs.ipynb)